In [0]:
%python
data = [
    (1, 100, 'aa', 67),
    (2, 200, 'bb', 36),
    (3, 300, 'cc', 54),
    (4, 400, 'dd', 18),
    (5, 500, 'ee', 20)
]

df = spark.createDataFrame(data, ['id', 'salary', 'name', 'age'])

In [0]:
%python
df.write \
    .format("delta") \
    .mode("append") \
    .save("/Volumes/datalake_jung/default/delta_volume/schema_level/")

## Enable Column Mapping

In [0]:
%sql
-- Enable column mapping on the Delta table
ALTER TABLE delta.`/Volumes/datalake_jung/default/delta_volume/schema_level/`
SET TBLPROPERTIES (
    'delta.minReaderVersion' = '2',                 -- The minimum Delta protocol version required to read this table.
    'delta.minWriterVersion' = '5',                 -- The minimum Delta protocol version required to write/change this table.
    'delta.columnMapping.mode' = 'name'              -- Delta gives each column a stable physical identity, separate from its logical name.
);

-- Rename the column
ALTER TABLE delta.`/Volumes/datalake_jung/default/delta_volume/schema_level/`
RENAME COLUMN name TO customer_name;

In [0]:
SELECT *
FROM delta.`/Volumes/datalake_jung/default/delta_volume/schema_level/`

id,salary,customer_name,age
1,100,aa,67
2,200,bb,36
3,300,cc,54
4,400,dd,18
5,500,ee,20


## Delta Log Level Schema Changes

In [0]:
%python
df_json = spark.read \
                .format("json") \
                .load("/Volumes/datalake_jung/default/delta_volume/schema_level/_delta_log/00000000000000000002.json")

display(df_json)

commitInfo,metaData
"List(0913-154046-8volzgta-v2n, false, Databricks-Runtime/19.6.x-aarch64-photon-scala2.13, true, WriteSerializable, List(1317694261002638), RENAME COLUMN, List(customer_name, name), 534a025e-f99c-46ed-b44d-745392a09b77, 1, 1789314135604, e09bfdaf-4bd1-440e-ae42-4eddc53144a0, 74660554949313, minhdung38109141@gmail.com)",null
null,"List(List(4, name, true), 1789307891048, List(parquet), bdc41c4f-0c96-42c4-9c48-852313b60af5, List(), {""type"":""struct"",""fields"":[{""name"":""id"",""type"":""long"",""nullable"":true,""metadata"":{""delta.columnMapping.id"":1,""delta.columnMapping.physicalName"":""id""}},{""name"":""salary"",""type"":""long"",""nullable"":true,""metadata"":{""delta.columnMapping.id"":2,""delta.columnMapping.physicalName"":""salary""}},{""name"":""customer_name"",""type"":""string"",""nullable"":true,""metadata"":{""delta.columnMapping.id"":3,""delta.columnMapping.physicalName"":""name""}},{""name"":""age"",""type"":""long"",""nullable"":true,""metadata"":{""delta.columnMapping.id"":4,""delta.columnMapping.physicalName"":""age""}}]})"
